# Importación de librerías

In [231]:
import pandas as pd
import sqlite3

# Ingesta de datos

In [188]:
archivo_excel = pd.ExcelFile("data\INSUMOS OPERACIONALES STOCK RELIX.xlsx")
print("Hojas disponibles:", archivo_excel.sheet_names)

Hojas disponibles: ['Insumos Operacionales 2025', 'INGRESO', 'SALIDA', 'STOCK ']


In [189]:
insumos = pd.read_excel("data\INSUMOS OPERACIONALES STOCK RELIX.xlsx", sheet_name='Insumos Operacionales 2025', skiprows=7)

In [190]:
insumos.head()

,Unnamed: 0,ITEM,CODIGO SAP/OTRO,DESCRIPCION DEL MATERIAL,CLASIFICACIÓN,U M,OBSERVACIONES,Unnamed: 7
0,NaN,1.1,11174821,DUCTO RECTO;PE100;PN20;280 MM;12 M;PL,PIPING,UN,NaN,NaN
1,NaN,1.2,11177716,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam",PIPING,UN,NaN,NaN
2,NaN,1.3,en catalogacion,"Tuberia HDPE 200MM DN, Flanges Moviles C150, c...",PIPING,UN,NaN,NaN
3,NaN,1.4,11151921,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam ...",PIPING,UN,NaN,NaN
4,NaN,1.5,11019704,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL",PIPING,UN,NaN,NaN


In [191]:
# Se eliminan columnas innecesarias dentro del archivo
insumos = insumos.drop(insumos.columns[[0, 7]], axis=1)
insumos.head()

,ITEM,CODIGO SAP/OTRO,DESCRIPCION DEL MATERIAL,CLASIFICACIÓN,U M,OBSERVACIONES
0,1.1,11174821,DUCTO RECTO;PE100;PN20;280 MM;12 M;PL,PIPING,UN,NaN
1,1.2,11177716,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam",PIPING,UN,NaN
2,1.3,en catalogacion,"Tuberia HDPE 200MM DN, Flanges Moviles C150, c...",PIPING,UN,NaN
3,1.4,11151921,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam ...",PIPING,UN,NaN
4,1.5,11019704,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL",PIPING,UN,NaN


In [192]:
insumos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 491 entries, 0 to 490
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ITEM                       490 non-null    float64
 1   CODIGO SAP/OTRO            490 non-null    object 
 2   DESCRIPCION DEL MATERIAL   490 non-null    object 
 3   CLASIFICACIÓN              490 non-null    object 
 4   U M                        486 non-null    object 
 5   OBSERVACIONES              0 non-null      float64
dtypes: float64(2), object(4)
memory usage: 23.1+ KB


In [193]:
insumos.columns

Index(['ITEM', 'CODIGO SAP/OTRO', 'DESCRIPCION DEL MATERIAL ',
       'CLASIFICACIÓN ', 'U M', 'OBSERVACIONES'],
      dtype='object')

# Transformación

In [194]:
# Se eliminan espacios en blanco de los nombres de las columnas
insumos.columns = insumos.columns.str.strip()
insumos.columns

Index(['ITEM', 'CODIGO SAP/OTRO', 'DESCRIPCION DEL MATERIAL', 'CLASIFICACIÓN',
       'U M', 'OBSERVACIONES'],
      dtype='object')

### Categorias unicas de la clasificación.

In [195]:
insumos['CLASIFICACIÓN'].unique()

array(['PIPING ', 'PIEZAS ESPECIALES', 'VALVULAS', 'HUMECTACIÓN ',
       'PIEZÓMETROS ', 'IMPERMEABILIAZACIÓN ', 'OBRAS CIVILES ',
       'SERVICIOS GENERALES', 'ILUMINACION EMPALIZADA', 'TALLER BERLIAM',
       nan, 'RIEGO MURO TRANQUE', 'INSTRUMENTACIÓN GEOTECNICA'],
      dtype=object)

In [196]:
insumos['CLASIFICACIÓN'] = insumos['CLASIFICACIÓN'].str.strip()
insumos['CLASIFICACIÓN'].unique()

array(['PIPING', 'PIEZAS ESPECIALES', 'VALVULAS', 'HUMECTACIÓN',
       'PIEZÓMETROS', 'IMPERMEABILIAZACIÓN', 'OBRAS CIVILES',
       'SERVICIOS GENERALES', 'ILUMINACION EMPALIZADA', 'TALLER BERLIAM',
       nan, 'RIEGO MURO TRANQUE', 'INSTRUMENTACIÓN GEOTECNICA'],
      dtype=object)

### Verificación valores nulos

In [197]:
# Revisión de valores nulos en la columna CLASIFICACIÓN
insumos[insumos['CLASIFICACIÓN'].isnull()]
# TODO: llenar clasificación NaN

,ITEM,CODIGO SAP/OTRO,DESCRIPCION DEL MATERIAL,CLASIFICACIÓN,U M,OBSERVACIONES
481,11.29,11220592,MAURO TUERCA STORZ 4 PULGADAS LINEA DE RIEGO,NaN,UN,NaN


In [198]:
insumos[insumos['CODIGO SAP/OTRO'].isnull()]
# TODO: Considerar iluminación empalizada como categoría?

,ITEM,CODIGO SAP/OTRO,DESCRIPCION DEL MATERIAL,CLASIFICACIÓN,U M,OBSERVACIONES
490,NaN,NaN,NaN,ILUMINACION EMPALIZADA,NaN,NaN


In [199]:
insumos[insumos['U M'].isnull()]
# TODO: Llenar unidades de medida faltantes


,ITEM,CODIGO SAP/OTRO,DESCRIPCION DEL MATERIAL,CLASIFICACIÓN,U M,OBSERVACIONES
166,4.40,catalogar sap,"Flange Ciego diam 28"" ANSI C150",PIEZAS ESPECIALES,NaN,NaN
167,4.50,catalogar sap,"Flange Ciego 28"" ANSI C300",PIEZAS ESPECIALES,NaN,NaN
348,9.95,catalogar sap,Tubos PVC Bastones Reflectantes,SERVICIOS GENERALES,NaN,NaN
352,9.99,11215216,BOQUILLAS DEPOSITACION 50MM POLIURETANO,SERVICIOS GENERALES,NaN,NaN
490,NaN,NaN,NaN,ILUMINACION EMPALIZADA,NaN,NaN


In [233]:
insumos['CODIGO SAP/OTRO'].value_counts()

CODIGO SAP/OTRO
catalogar sap      10
en catalogacion     9
44001452            2
11184744            2
11021056            2
                   ..
11188680            1
11188679            1
11188678            1
11188677            1
11188676            1
Name: count, Length: 470, dtype: int64

In [200]:
insumos['U M'].unique()
# TODO: Corregir UM inconsistentes

array(['UN', nan, 'kit', 'Bins', 'M2', 'ROLLOS', 'M3', 'M', 'kg', 'Mt',
       'Kg', 'Metros', 'KIT', 'lts', 'mt', 'Unidad', 'Bin', 'Rollo',
       'saco 25 kg', 'Kit'], dtype=object)

In [201]:
insumos.head(3)

,ITEM,CODIGO SAP/OTRO,DESCRIPCION DEL MATERIAL,CLASIFICACIÓN,U M,OBSERVACIONES
0,1.1,11174821,DUCTO RECTO;PE100;PN20;280 MM;12 M;PL,PIPING,UN,NaN
1,1.2,11177716,"DUCTO;HDPE;12MT;PE100;PN12,5;F01;PL - 8"" diam",PIPING,UN,NaN
2,1.3,en catalogacion,"Tuberia HDPE 200MM DN, Flanges Moviles C150, c...",PIPING,UN,NaN


In [202]:
insumos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 491 entries, 0 to 490
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ITEM                      490 non-null    float64
 1   CODIGO SAP/OTRO           490 non-null    object 
 2   DESCRIPCION DEL MATERIAL  490 non-null    object 
 3   CLASIFICACIÓN             490 non-null    object 
 4   U M                       486 non-null    object 
 5   OBSERVACIONES             0 non-null      float64
dtypes: float64(2), object(4)
memory usage: 23.1+ KB
